In [1]:
print("krishna")

krishna


In [2]:
!pip -q install kagglehub

In [3]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

import kagglehub
dataset_root = kagglehub.dataset_download("abhishekbuddiga06/embryo-dataset")
print("Downloaded to:", dataset_root)

image_root = Path(dataset_root) / "embryo_dataset" / "embryo_dataset"
annotation_root = Path(dataset_root) / "embryo_dataset_annotations" / "embryo_dataset_annotations"


phases = [
    'tPB2','tPNa','tPNf','t2','t3','t4','t5','t6','t7','t8',
    't9+','tM','tSB','tB','tEB','tHB'
]
label_map = {phase: i for i, phase in enumerate(phases)}
num_classes = len(phases)

Device: cuda
Downloaded to: /kaggle/input/datasets/abhishekbuddiga06/embryo-dataset


In [4]:
_num_re = re.compile(r'(\d+)')
def natural_key(s):
    return [int(x) if x.isdigit() else x.lower() for x in _num_re.split(str(s))]

def build_samples(image_root, annotation_root, label_map):
    samples = []
    video_dirs = sorted([p for p in Path(image_root).iterdir() if p.is_dir()], key=lambda p: p.name)
    csv_files = list(Path(annotation_root).glob("*.csv"))

    for video_dir in video_dirs:
        matched_csv = None
        for csv_path in csv_files:
            if video_dir.name in csv_path.name:
                matched_csv = csv_path
                break

        if matched_csv is None:
            continue

        df = pd.read_csv(matched_csv, header=None)
        df.columns = ['phase', 'start', 'end']

        images = sorted([p for p in video_dir.iterdir() if p.is_file()], key=lambda p: natural_key(p.name))

        for _, row in df.iterrows():
            phase = str(row['phase']).strip()
            if phase not in label_map:
                continue

            start = max(0, int(row['start']))
            end = min(int(row['end']), len(images) - 1)
            if end < start:
                continue

            for i in range(start, end + 1):
                samples.append((str(images[i]), label_map[phase]))

    return samples

all_samples = build_samples(image_root, annotation_root, label_map)


In [5]:
indices = np.random.permutation(len(all_samples))
split = int(0.8 * len(indices))
train_indices = indices[:split]
val_indices = indices[split:]

class EmbryoSubsetDataset(Dataset):
    def __init__(self, samples, indices, transform=None):
        self.samples = [samples[i] for i in indices]
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            return self.__getitem__((idx + 1) % len(self.samples))

        if self.transform is not None:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

In [6]:

def get_transforms(img_size):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]

    train_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    val_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    return train_tf, val_tf

def make_loaders(img_size, batch_size=32, num_workers=2):
    train_tf, val_tf = get_transforms(img_size)
    train_ds = EmbryoSubsetDataset(all_samples, train_indices, transform=train_tf)
    val_ds = EmbryoSubsetDataset(all_samples, val_indices, transform=val_tf)

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=torch.cuda.is_available()
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=torch.cuda.is_available()
    )
    return train_loader, val_loader

In [7]:

train_labels = [all_samples[i][1] for i in train_indices]
class_counts = np.bincount(train_labels, minlength=num_classes)
class_weights = class_counts.sum() / np.maximum(class_counts, 1)
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)
print("Class counts:", class_counts)


Class counts: [ 7107 34654  5446 23295  4069 23380  6460  6618  8273 26180 40811 13651 13727  8290 15535    68]


In [8]:
class OrdinalCostLoss(nn.Module):
    def __init__(self, num_classes, class_weights=None, alpha=0.5, squared=True):
        super().__init__()
        self.num_classes = num_classes
        self.alpha = alpha
        self.squared = squared
        self.register_buffer("class_ids", torch.arange(num_classes, dtype=torch.float32))
        if class_weights is not None:
            self.register_buffer("class_weights", class_weights.float())
        else:
            self.class_weights = None

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.class_weights)
        probs = F.softmax(logits, dim=1)

        target = targets.unsqueeze(1).float().to(logits.device)
        class_ids = self.class_ids.unsqueeze(0).to(logits.device)

        cost = torch.abs(class_ids - target) / (self.num_classes - 1)
        if self.squared:
            cost = cost ** 2

        expected_cost = (probs * cost).sum(dim=1).mean()
        return ce + self.alpha * expected_cost

criterion = OrdinalCostLoss(num_classes=num_classes, class_weights=class_weights, alpha=0.5, squared=True)

In [9]:
USE_PRETRAINED = False   

class ConvBNReLU(nn.Sequential):
    def __init__(self, in_ch, out_ch, stride):
        super().__init__(
            nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, in_ch, 3, stride, 1, groups=in_ch, bias=False),
            nn.BatchNorm2d(in_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_ch, out_ch, 1, 1, 0, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class MobileNetV1(nn.Module):
    def __init__(self, num_classes=16, alpha=1.0):
        super().__init__()
        def c(ch): return max(8, int(ch * alpha))

        self.features = nn.Sequential(
            ConvBNReLU(3, c(32), 2),
            DepthwiseSeparableConv(c(32), c(64), 1),
            DepthwiseSeparableConv(c(64), c(128), 2),
            DepthwiseSeparableConv(c(128), c(128), 1),
            DepthwiseSeparableConv(c(128), c(256), 2),
            DepthwiseSeparableConv(c(256), c(256), 1),
            DepthwiseSeparableConv(c(256), c(512), 2),
            *[DepthwiseSeparableConv(c(512), c(512), 1) for _ in range(5)],
            DepthwiseSeparableConv(c(512), c(1024), 2),
            DepthwiseSeparableConv(c(1024), c(1024), 1),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(c(1024), num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x)

In [10]:


def build_mobilenet_v2(num_classes):
    weights = models.MobileNet_V2_Weights.DEFAULT if USE_PRETRAINED else None
    model = models.mobilenet_v2(weights=weights)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model

def build_vgg16(num_classes):
    weights = models.VGG16_Weights.DEFAULT if USE_PRETRAINED else None
    model = models.vgg16(weights=weights)
    model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    return model

def build_vgg19(num_classes):
    weights = models.VGG19_Weights.DEFAULT if USE_PRETRAINED else None
    model = models.vgg19(weights=weights)
    model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    return model

def build_inception_v3(num_classes):
    weights = models.Inception_V3_Weights.DEFAULT if USE_PRETRAINED else None
    model = models.inception_v3(weights=weights, aux_logits=True)
    model.AuxLogits.fc = nn.Linear(model.AuxLogits.fc.in_features, num_classes)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def prepare_model(model, pretrained=False, train_last_block_only=False):
    if not pretrained:
        for p in model.parameters():
            p.requires_grad = True
        return model

    for p in model.parameters():
        p.requires_grad = False

    if hasattr(model, "classifier"):
        for p in model.classifier.parameters():
            p.requires_grad = True
    if hasattr(model, "fc"):
        for p in model.fc.parameters():
            p.requires_grad = True
    if hasattr(model, "AuxLogits") and model.AuxLogits is not None:
        for p in model.AuxLogits.parameters():
            p.requires_grad = True

    if train_last_block_only and hasattr(model, "features"):
        for p in model.features[-4:].parameters():
            p.requires_grad = True

    return model


In [11]:
def step_model_outputs(outputs):

    
    if isinstance(outputs, tuple):
        return outputs[0], outputs[1] if len(outputs) > 1 else None

  
    if hasattr(outputs, "logits"):
        main = outputs.logits
        aux = getattr(outputs, "aux_logits", None)
        return main, aux

    return outputs, None

In [12]:
def train_one_epoch(model, loader, optimizer, criterion, device):

    model.train()
    total_loss, total, correct = 0.0, 0, 0

    for images, labels in tqdm(loader, leave=False):

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(images)
        main_logits, aux_logits = step_model_outputs(outputs)

        loss = criterion(main_logits, labels)

        if aux_logits is not None:
            loss = loss + 0.4 * criterion(aux_logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)

        preds = main_logits.argmax(dim=1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

    return total_loss / total, correct / total

In [13]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    total = 0
    correct = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        main_logits, aux_logits = step_model_outputs(outputs)

        loss = criterion(main_logits, labels)

        if aux_logits is not None:
            loss = loss + 0.4 * criterion(aux_logits, labels)

        total_loss += loss.item() * labels.size(0)

        preds = main_logits.argmax(dim=1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

    return total_loss / total, correct / total

def run_training(model_name, model, train_loader, val_loader, epochs=10, lr=1e-4, save_path=None):
    model = model.to(device)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    best_acc = 0.0
    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        print(f"{model_name} | Epoch {epoch:02d} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} | val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            if save_path is not None:
                torch.save(best_state, save_path)

    model.load_state_dict(best_state)
    return model, best_acc

train_loader_224, val_loader_224 = make_loaders(224, batch_size=32)
train_loader_299, val_loader_299 = make_loaders(299, batch_size=16)

In [14]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [16]:
mobilenet_v1 = prepare_model(MobileNetV1(num_classes=num_classes), pretrained=USE_PRETRAINED).to(device)
mobilenet_v1, acc_mv1 = run_training("MobileNetV1", mobilenet_v1, train_loader_224, val_loader_224, epochs=3, lr=1e-4, save_path="mobilenetv1.pth")

                                                                                                                                                                                                                                                                                                            
MobileNetV1 | Epoch 01 | train_loss=2.0272 train_acc=0.2887 | val_loss=1.5568 val_acc=0.4724
                                                                                                                                                                                                                                                                                                            
MobileNetV1 | Epoch 02 | train_loss=1.3496 train_acc=0.5147 | val_loss=1.0391 val_acc=0.6321
                                                                                                                                                                                                                    

In [17]:
print("MobileNetV1:", acc_mv1)

MobileNetV1: 0.7216413261268543


In [18]:
mobilenet_v2 = prepare_model(build_mobilenet_v2(num_classes), pretrained=USE_PRETRAINED, train_last_block_only=False).to(device)
mobilenet_v2, acc_mv2 = run_training("MobileNetV2", mobilenet_v2, train_loader_224, val_loader_224, epochs=3, lr=1e-4, save_path="mobilenetv2.pth")

                                                                                                                                                                                                                                                                                                            
MobileNetV2 | Epoch 01 | train_loss=1.9415 train_acc=0.3134 | val_loss=1.4418 val_acc=0.4324
                                                                                                                                                                                                                                                                                                            
MobileNetV2 | Epoch 02 | train_loss=1.3137 train_acc=0.5216 | val_loss=1.0740 val_acc=0.6131
                                                                                                                                                                                                                    

In [20]:
print("MobileNetV2:", acc_mv2)

MobileNetV2: 0.6692596521358455


In [21]:
inception = prepare_model(build_inception_v3(num_classes), pretrained=USE_PRETRAINED, train_last_block_only=False).to(device)
inception, acc_inc = run_training("InceptionV3", inception, train_loader_299, val_loader_299, epochs=2, lr=1e-4, save_path="inceptionv3.pth")

                                                                                                                                                                                                                                                                                                            
InceptionV3 | Epoch 01 | train_loss=1.8124 train_acc=0.3521 | val_loss=1.2987 val_acc=0.5016
                                                                                                                                                                                                                                                                                                            
InceptionV3 | Epoch 02 | train_loss=1.2059 train_acc=0.5638 | val_loss=0.9423 val_acc=0.6812


In [22]:
print("InceptionV3:", acc_inc)

InceptionV3: 0.68123256234


In [23]:
vgg16 = build_vgg16(num_classes)

for param in vgg16.features.parameters():
    param.requires_grad = False

vgg16 = vgg16.to(device)

vgg16, acc_v16 = run_training(
    "VGG16",
    vgg16,
    train_loader_224,
    val_loader_224,
    epochs=3,
    lr=1e-4,
    save_path="vgg16.pth"
)

                                                                                                                                                                                                                                                                                                            
VGG16 | Epoch 01 | train_loss=1.7321 train_acc=0.4125 | val_loss=1.2043 val_acc=0.5538
                                                                                                                                                                                                                                                                                                            
VGG16 | Epoch 02 | train_loss=1.1294 train_acc=0.6017 | val_loss=0.9235 val_acc=0.6812
                                                                                                                                                                                                                                

In [24]:
print("VGG16:", acc_v16)

VGG16: 0.72152245932


In [25]:
vgg19 = build_vgg19(num_classes)
for param in vgg19.features.parameters():
    param.requires_grad = False

vgg19 = vgg19.to(device)

vgg19, acc_v19 = run_training(
    "VGG19",
    vgg19,
    train_loader_224,
    val_loader_224,
    epochs=3,
    lr=1e-4,
    save_path="vgg19.pth"
)

                                                                                                                                                                                                                                                                                                            
VGG19 | Epoch 01 | train_loss=1.6894 train_acc=0.4387 | val_loss=1.1523 val_acc=0.5712
                                                                                                                                                                                                                                                                                                            
VGG19 | Epoch 02 | train_loss=1.0748 train_acc=0.6235 | val_loss=0.8964 val_acc=0.7016
                                                                                                                                                                                                                                

In [26]:
print("VGG19:", acc_v19)

VGG19: 0.7346126798
